## Setup

Importings libs and setting up the data frames...

In [61]:
# Import necessary libraries
import os
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
from enum import IntFlag

plt.style.use('ggplot')
%matplotlib inline

In [62]:
path = r"C:\Users\Kuinox\Documents\parquet_output"
addresses_file = os.path.join(path, 'addresses.parquet')
symbols_file = os.path.join(path, 'symbols.parquet')
tracesamples_file = os.path.join(path, 'tracesamples.parquet')
aux_events_file = os.path.join(path, 'aux_events.parquet')

In [63]:
addresses_df = pl.scan_parquet(addresses_file).lazy()
symbols_df = pl.scan_parquet(symbols_file).lazy()
tracesamples_df = pl.scan_parquet(tracesamples_file).lazy()
aux_events_df = pl.scan_parquet(aux_events_file).lazy()


### How the raw data looks like...

In [64]:
print("\n--- Addresses Schema ---")
display(addresses_df.collect_schema())

print("\n--- Symbols Schema ---")
display(symbols_df.collect_schema())

print("\n--- Trace Samples Schema ---")
display(tracesamples_df.collect_schema())

print("\n--- Aux Events Schema ---")
display(aux_events_df.collect_schema())


addresses_count = addresses_df.select(pl.len()).collect().item()
symbols_count = symbols_df.select(pl.len()).collect().item()
tracesamples_count = tracesamples_df.select(pl.len()).collect().item()
aux_events_count = aux_events_df.select(pl.len()).collect().item()

print(f"Addresses count: {addresses_count:,}")
print(f"Symbols count: {symbols_count:,}")
print(f"Trace samples count: {tracesamples_count:,}")
print(f"Aux events count: {aux_events_count:,}")


--- Addresses Schema ---


Schema([('id', UInt64),
        ('traceId', UInt64),
        ('address', UInt64),
        ('pid', UInt32),
        ('isIp', Boolean),
        ('size', UInt32),
        ('symoff', UInt32),
        ('symStrId', UInt64),
        ('symStart', UInt64),
        ('symEnd', UInt64),
        ('dso', UInt64),
        ('symBinding', UInt8),
        ('is64Bit', UInt8),
        ('isKernelIp', UInt8),
        ('buildId', Binary),
        ('filtered', UInt8),
        ('comm', UInt64),
        ('priv', UInt64)])


--- Symbols Schema ---


Schema([('id', UInt64), ('symbol', String)])


--- Trace Samples Schema ---


Schema([('id', UInt64),
        ('perfId', UInt64),
        ('pid', UInt32),
        ('tid', UInt32),
        ('time', UInt64),
        ('flags', UInt32),
        ('cpu', UInt32),
        ('ip', UInt64),
        ('addr', UInt64),
        ('period', UInt64),
        ('insnCnt', UInt64),
        ('cycCnt', UInt64),
        ('weight', UInt64),
        ('cpumode', UInt8),
        ('addrCorrelatesSym', UInt8),
        ('event', String),
        ('machinePid', UInt32),
        ('vcpu', UInt32)])


--- Aux Events Schema ---


Schema([('time', UInt64),
        ('pid', UInt64),
        ('tid', UInt64),
        ('cpu', UInt64),
        ('flags', UInt64)])

Addresses count: 232,917,414
Symbols count: 720
Trace samples count: 212,496,770
Aux events count: 897


## Pre Processing 
The purpose of the tool we used before are just to to quickly collect and pack the data in a format accessible.  
Also, it allow you to work on this even if you don't run linux, bare metal, with an intel cpu.

### Checking assumptions

In [ ]:
# are all instruction ordered by time ?
time_order_check = tracesamples_df.select(
    pl.col('time').diff().lt(0).sum().alias('out_of_order_count')
).collect()

out_of_order_count = time_order_check[0, 'out_of_order_count']

if out_of_order_count == 0:
    print("All instructions are correctly ordered by time.")
else:
    print(f"Found {out_of_order_count} instances where instructions are not ordered by time.")
    raise ValueError("Time sequence error detected in trace samples. Processing halted.")

All instructions are correctly ordered by time.


### Aux Events

While the other events happen from the DLFilter, this one come from directly parsing the file.  
Why ? `perf` need to run on the target machine and allow us to fetch easily symbols infos from an instruction. But it only pass samples to the dlfilter.  
There is an important informations in the events: AUX data loss infos.  
This tell us when intel_pt buffer got saturated and it had to drop events.

In [65]:

aux_events_df = aux_events_df.with_columns(
    (pl.col('flags') != 0).alias('is_error')
)

aux_data_loss = aux_events_df.filter(pl.col('is_error'))
aux_data_loss_count = aux_data_loss.select(pl.len()).collect().item()
print(f"Aux data loss count: {aux_data_loss_count}")

Aux data loss count: 13


### Samples

In [66]:
class SampleFlags(IntFlag):
    BRANCH       = 1 << 0
    CALL         = 1 << 1
    RETURN       = 1 << 2
    CONDITIONAL  = 1 << 3
    SYSCALLRET   = 1 << 4
    ASYNC        = 1 << 5
    INTERRUPT    = 1 << 6
    TX_ABORT     = 1 << 7
    TRACE_BEGIN  = 1 << 8
    TRACE_END    = 1 << 9
    IN_TX        = 1 << 10
    VMENTRY      = 1 << 11
    VMEXIT       = 1 << 12

tracesamples_df = tracesamples_df.with_columns([
        (pl.col('flags') & SampleFlags.BRANCH > 0).alias('is_branch'),
        (pl.col('flags') & SampleFlags.CALL > 0).alias('is_call'),
        (pl.col('flags') & SampleFlags.RETURN > 0).alias('is_return'),
        (pl.col('flags') & SampleFlags.CONDITIONAL > 0).alias('is_conditional'),
        (pl.col('flags') & SampleFlags.SYSCALLRET > 0).alias('is_syscallret'),
        (pl.col('flags') & SampleFlags.ASYNC > 0).alias('is_async'),
        (pl.col('flags') & SampleFlags.INTERRUPT > 0).alias('is_interrupt'),
        (pl.col('flags') & SampleFlags.TX_ABORT > 0).alias('is_tx_abort'),
        (pl.col('flags') & SampleFlags.TRACE_BEGIN > 0).alias('is_trace_begin'),
        (pl.col('flags') & SampleFlags.TRACE_END > 0).alias('is_trace_end'),
        (pl.col('flags') & SampleFlags.IN_TX > 0).alias('is_in_tx'),
        (pl.col('flags') & SampleFlags.VMENTRY > 0).alias('is_vmentry'),
        (pl.col('flags') & SampleFlags.VMEXIT > 0).alias('is_vmexit')
])

tracesamples_df = tracesamples_df.with_columns([
    pl.col("tid").alias("threadId"),
    pl.col("pid").alias("processId"),
    pl.col("insnCnt").alias("instructionCount"),
    pl.col("cycCnt").alias("cycleCount"),
    pl.col("addrCorrelatesSym").alias("addressCorrelatesSymbol")
])

flag_infos = (
    tracesamples_df
    .select([
        pl.sum('is_branch').alias('branch_count'),
        pl.sum('is_call').alias('call_count'),
        pl.sum('is_return').alias('return_count'),
        pl.sum('is_conditional').alias('conditional_count'),
        pl.sum('is_syscallret').alias('syscallret_count'),
        pl.sum('is_async').alias('async_count'),
        pl.sum('is_interrupt').alias('interrupt_count'),
        pl.sum('is_tx_abort').alias('tx_abort_count'),
        pl.sum('is_trace_begin').alias('trace_begin_count'),
        pl.sum('is_trace_end').alias('trace_end_count'),
        pl.sum('is_in_tx').alias('in_tx_count'),
        pl.sum('is_vmentry').alias('vmentry_count'),
        pl.sum('is_vmexit').alias('vmexit_count')
    ])
    .collect()
)
instruction_counts = tracesamples_df.select(pl.len()).collect().item()

print(f"Instruction counts: {instruction_counts}")

print("Branch type counts:")
display(flag_infos)

threads_df = tracesamples_df.group_by("threadId")

thread_instruction_counts = threads_df.agg(pl.count('instructionCount').alias('instruction_count')).collect()
print("Number of instructions per thread:")
display(thread_instruction_counts)

aux_data_loss_times = aux_data_loss.select(pl.col('time')).collect()

split_samples = []
prev_time = 0

for time in aux_data_loss_times['time']:
    split_samples.append(
        tracesamples_df.filter((pl.col('time') > prev_time) & (pl.col('time') <= time)).collect()
    )
    prev_time = time

# Add the remaining samples after the last aux data loss
split_samples.append(
    tracesamples_df.filter(pl.col('time') > prev_time).collect()
)

for i, sample in enumerate(split_samples):
    print(f"Sample {i + 1}: {sample.shape[0]:,} instructions")

Instruction counts: 212496770
Branch type counts:


branch_count,call_count,return_count,conditional_count,syscallret_count,async_count,interrupt_count,tx_abort_count,trace_begin_count,trace_end_count,in_tx_count,vmentry_count,vmexit_count
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
821738,212470735,136304216,0,0,0,0,0,0,0,0,0,0


Number of instructions per thread:


threadId,instruction_count
u32,u32
18466,10201
18463,4571
18469,1708
18464,5941
18467,804707
18461,211657182
18468,8185
18465,4275


Sample 1: 1,675,576 instructions
Sample 2: 3,663,544 instructions
Sample 3: 466,397 instructions
Sample 4: 977,340 instructions
Sample 5: 501,640 instructions
Sample 6: 2,494,973 instructions
Sample 7: 2,181,726 instructions
Sample 8: 2,114,664 instructions
Sample 9: 1,454,041 instructions
Sample 10: 664,542 instructions
Sample 11: 135,921,358 instructions
Sample 12: 2,653,994 instructions
Sample 13: 57,192,784 instructions
Sample 14: 534,191 instructions


### Stack Reconstruction

Now we can reconstruct the stack for each instruction series.

### 